# Model 2: Pothole severity classifier

Takes a close-up crop of one pothole and predicts its severity: `1_minor`, `2_moderate` or `3_severe`. This is stage 2 of the pipeline (stage 1 is the YOLO detector in `train.ipynb`).

**Approach:** ResNet18 pretrained on ImageNet, last layer replaced with 3 outputs, `layer4` unfrozen for fine-tuning (learning rates: `fc` 0.001, `layer4` 0.0001), 10 epochs. The epoch with the best validation accuracy is saved to `severity_layer4.pt`.

**Dataset** (not included in the repo): 1,311 crops, made from Kaggle pothole crops plus automatic crops from the YOLO detector, sorted by hand into 3 folders (457 minor, 434 moderate, 420 severe). Split 70/15/15 with seed 42.

**Result:** 79% accuracy on 199 test images (frozen-backbone version: 72 to 75%). Moderate is the weakest class and is mostly confused with severe.

**Limitations**
- Small dataset, and the severity labels come from one person, so "moderate" is subjective.
- The model overfits somewhat (train accuracy about 96% vs about 79% test).
- The split is random by crop, not by source photo, so crops of the same photo may land in both train and test and the test score could be a bit optimistic.
- Single training run, no repeated experiments.

In [1]:
import os
import shutil
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report

print("GPU available:", torch.cuda.is_available())

GPU available: True


## 1. Count the images

In [2]:
for tier in ["1_minor", "2_moderate", "3_severe"]:
    folder = os.path.join("severity_dataset", tier)
    print(tier, len(os.listdir(folder)), "images")

1_minor 457 images
2_moderate 434 images
3_severe 420 images


## 2. Split into train / val / test (70 / 15 / 15, seed 42)

In [3]:
random.seed(42)

source_dir = "severity_dataset"
split_dir = "severity_dataset_split"
tiers = ["1_minor", "2_moderate", "3_severe"]

shutil.rmtree(split_dir, ignore_errors=True)

for tier in tiers:
    images = os.listdir(os.path.join(source_dir, tier))
    random.shuffle(images)

    n = len(images)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)

    groups = {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train + n_val:],
    }

    for split_name, files in groups.items():
        dest = os.path.join(split_dir, split_name, tier)
        os.makedirs(dest, exist_ok=True)
        for f in files:
            shutil.copy2(os.path.join(source_dir, tier, f), os.path.join(dest, f))

    print(tier, len(groups["train"]), "train,", len(groups["val"]), "val,", len(groups["test"]), "test")

1_minor 319 train, 68 val, 70 test
2_moderate 303 train, 65 val, 66 test
3_severe 294 train, 63 val, 63 test


## 3. Load the images (transforms, ImageFolder, DataLoader)

In [4]:
data_dir = "severity_dataset_split"

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(f"{data_dir}/train", transform=train_transforms)
val_dataset = datasets.ImageFolder(f"{data_dir}/val", transform=eval_transforms)
test_dataset = datasets.ImageFolder(f"{data_dir}/test", transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(train_dataset.classes)
print(len(train_dataset), len(val_dataset), len(test_dataset))

['1_minor', '2_moderate', '3_severe']
916 196 199


## 4. Build the model
Fresh pretrained ResNet18, every layer frozen, last layer replaced with a new 3-output layer.

In [5]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model.fc)
print("Device:", device)

Linear(in_features=512, out_features=3, bias=True)
Device: cuda


## 5. Unfreeze layer4 and set up the optimizer
First I trained with every layer frozen and only the new fc layer learning. Test accuracy was 72 to 75%, which was too low. I then unfroze layer4, the last block, so it could adapt to pothole-specific details, and accuracy rose to 79%. fc is newly created (random), so it learns faster (lr 0.001); layer4 is already pretrained, so it uses a smaller rate (0.0001) to avoid overwriting useful features. These rates are common starting values and were not tuned.

In [ ]:
for param in model.layer4.parameters():
    param.requires_grad = True

optimizer = optim.Adam([
    {"params": model.fc.parameters(), "lr": 0.001},
    {"params": model.layer4.parameters(), "lr": 0.0001},
])

criterion = nn.CrossEntropyLoss()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(trainable, "trainable numbers") 

8395267 trainable numbers


## 6. Train
Note: this saves the best epoch to `severity_layer4.pt` and overwrites the old file. Copy the file first if you want to keep the earlier weights.

In [7]:
num_epochs = 10
best_val_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()

    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()

    train_loss /= len(train_dataset)
    train_acc = train_correct / len(train_dataset)
    val_loss /= len(val_dataset)
    val_acc = val_correct / len(val_dataset)

    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "severity_layer4.pt")

Epoch 1/10 - Train Loss: 0.6699, Train Acc: 0.6998 - Val Loss: 0.3938, Val Acc: 0.8265
Epoch 2/10 - Train Loss: 0.4099, Train Acc: 0.8275 - Val Loss: 0.3832, Val Acc: 0.8520
Epoch 3/10 - Train Loss: 0.3364, Train Acc: 0.8603 - Val Loss: 0.3430, Val Acc: 0.8520
Epoch 4/10 - Train Loss: 0.2611, Train Acc: 0.8985 - Val Loss: 0.3783, Val Acc: 0.8418
Epoch 5/10 - Train Loss: 0.2260, Train Acc: 0.9138 - Val Loss: 0.4462, Val Acc: 0.8214
Epoch 6/10 - Train Loss: 0.1630, Train Acc: 0.9323 - Val Loss: 0.4658, Val Acc: 0.8265
Epoch 7/10 - Train Loss: 0.1502, Train Acc: 0.9443 - Val Loss: 0.4453, Val Acc: 0.8163
Epoch 8/10 - Train Loss: 0.1846, Train Acc: 0.9378 - Val Loss: 0.5237, Val Acc: 0.8316
Epoch 9/10 - Train Loss: 0.1191, Train Acc: 0.9640 - Val Loss: 0.5554, Val Acc: 0.8214
Epoch 10/10 - Train Loss: 0.1269, Train Acc: 0.9574 - Val Loss: 0.5168, Val Acc: 0.8265


## 7. Test the saved best model
Loads the best epoch and scores it on the 199 test images, which were never used to choose anything.

In [8]:
model.load_state_dict(torch.load("severity_layer4.pt", map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        all_preds.extend(outputs.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))
print(confusion_matrix(all_labels, all_preds))

              precision    recall  f1-score   support

     1_minor       0.82      0.91      0.86        70
  2_moderate       0.74      0.68      0.71        66
    3_severe       0.82      0.78      0.80        63

    accuracy                           0.79       199
   macro avg       0.79      0.79      0.79       199
weighted avg       0.79      0.79      0.79       199

[[64  6  0]
 [10 45 11]
 [ 4 10 49]]


How to read this. Each row is the true class and each column is the predicted class in the matrix, so the diagonal (64, 45, 49) holds the correct predictions: 158 of 199, which is 79% accuracy. With 3 balanced classes, guessing would give about 33%, so 79% means the model learned real patterns.

Precision is “when the model says this class, how often is it right”. Recall is “of the real examples of this class, how many did it find”. F1 combines both, and support is the number of test images per class.

Minor is the easiest class (recall 0.91, F1 0.86). Severe is next (F1 0.80). Moderate is the weakest (recall 0.68, F1 0.71): of 66 real moderate potholes, 10 were called minor and 11 were called severe.

The errors are mostly between neighbouring levels. A minor pothole was never called severe, and only 4 severe ones were called minor. This is expected, because the boundaries between minor/moderate and moderate/severe are subjective, and the labels come from one person.